# Alice asks the hub a question

*A walkthrough of the SyftHub Python SDK, v2, as a mock. Everything below runs against in-memory fakes that return
exactly what the real Hub and Space APIs return. Names, prices and documents are invented.*

Alice is an oncology researcher. She wants to know what adverse events were reported in recent phase 3 trials, across
data other people hold, and then talk it through with a model. She has never used SyftHub before.

In [1]:
import syfthub
from syfthub import AsyncHub, Options
from syfthub.testing import World, MockTransport

world = World()                                     # the mock's stand-in for the outside world
hub = AsyncHub(options=Options(http_client=MockTransport(world)))
await hub.login(username="alice", password="secret")

Hub wallet,"$1.50 (MPP only, experimental)"


## What is out there?

She starts by looking. Sources and models are listed together; the pricing column tells her what each one costs before
she touches it, and the policies column flags anything else that might get in the way.

In [2]:
eps = await hub.endpoints.list()
eps

endpoint,type,pricing,policies,about,★
bob/llama-3Llama 3 70B,model,free,,"General model, hosted by Bob. Free.",88
carol/papersOncology Trial Papers,data_source,free,,"Indexed phase 2/3 oncology publications, 2018-2026, with adverse-event tables.",42
erin/trialsTrial Registry Mirror,data_source,$0.05/request via stripe,,Structured registry records with adverse-event tables.,31
olga/trials-assistantTrials Assistant,model_data_source,free,,Answers over Olga's curated trial corpus and returns the passages it used.,27
lena/sahabat-aiSahabat-AI 8B,model,Rp300/request via xendit,,"Small bilingual model, hosted by Lena. Billed in IDR from the same wallet as her registry.",23
ivan/gpt-miniGPT Mini (metered),model,$0.02/request via stripe,,"Small fast model, prepaid per message.",23
dave/notesClinician Notes (de-identified),data_source,$0.02/document via xendit,rate_limit,De-identified progress notes from two teaching hospitals.,17
heidi/shared-corpusConsortium Corpus,data_source,$0.01/request via cluster,,Shared corpus billed through the consortium's managed wallet.,12
dave/imagingRadiology Reports (de-identified),data_source,$0.03/request via xendit,,Same wallet as dave/notes: one top-up funds both.,9
lena/id-registryIndonesian Trial Registry,data_source,Rp300/request via xendit,,Billed in IDR: a separate wallet even though the rail is xendit too.,6


Too many. She describes what she is after, roughly, and narrows to data sources. Then she adds two by name: Dave's
clinician notes, and Olga's assistant, an endpoint that answers in its own words *and* returns the passages it used.
An endpoint decides what it returns: references, a summary, or both. A search accepts any of them.

In [3]:
picked = eps.matching("trial adverse events").filter(type="data_source") + eps.pick("dave/notes", "olga/trials-assistant")
picked

endpoint,type,pricing,policies,about,★
carol/papersOncology Trial Papers,data_source,free,,"Indexed phase 2/3 oncology publications, 2018-2026, with adverse-event tables.",42
erin/trialsTrial Registry Mirror,data_source,$0.05/request via stripe,,Structured registry records with adverse-event tables.,31
frank/registryPharmacovigilance Feed,data_source,free,rate_limit,Adverse-event signal reports; heavily rate limited.,5
dave/notesClinician Notes (de-identified),data_source,$0.02/document via xendit,rate_limit,De-identified progress notes from two teaching hospitals.,17
olga/trials-assistantTrials Assistant,model_data_source,free,,Answers over Olga's curated trial corpus and returns the passages it used.,27


One of them is paid per document. She looks closer.

In [4]:
await hub.endpoints.get("dave/notes")

Endpoint(dave/notes, data_source, $0.02/document via xendit)

## What would this cost?

She writes the question and composes a search over the four sources. **Nothing is sent yet.** Pre-flight shows what
each call would cost at most, her balance on every wallet involved, and a verdict per source. Dave's notes are held:
she has no credits on his wallet.

In [5]:
plan = hub.search("What adverse events were reported in phase 3 trials?", sources=picked)
await plan.preflight()

SearchPlan "What adverse events were reported in phase 3 trials?"  est. max $0.05  (4 of 5 will be sent)
  carol/papers             free                             max $0.00    ready
  erin/trials              $0.05/request via stripe         max $0.05    ready
  frank/registry           free                             max $0.00    ready
  dave/notes               $0.02/document via xendit        max $0.10    needs credits  (held)
  olga/trials-assistant    free                             max $0.00    ready

The card tells her what to do. Credits are bought from Dave's Space; `top_up` returns the invoice with its checkout
link. She pays in the browser, the provider tells Dave's Space, and the balance is there.

In [6]:
invoice = await plan.top_up("dave/notes", "starter")
invoice

wallet,w-dave-xendit-usd
bundle,starter · $5.00
checkout,https://checkout.example/pay/inv-97672


In [7]:
world.pay(invoice)                                        # mock: Alice pays at the link and the webhook fires
await plan.refresh()

endpoint,pricing,max cost,your balance,pre-flight,
carol/papers,free,-,-,ready,
erin/trials,$0.05/request via stripe,$0.05,$1.00,ready,deducted from wallet erin stripe USD
frank/registry,free,-,-,ready,rate limit 2/m per_user
dave/notes,$0.02/document via xendit,$0.10,$5.00,ready,deducted from wallet dave xendit USD rate limit 60/m per_user
olga/trials-assistant,free,-,-,ready,


## Run it

`execute()` re-checks pre-flight, then asks every endpoint that passes, in parallel. Each row is what that Space
returned: the **returns** column says whether it was references, a summary, or both. Frank's feed is rate limited
today; that could not be known in advance, so it was sent and came back rejected. The charges table at the bottom is
assembled from the Spaces' own receipts.

In [8]:
results = await plan.execute()
results

endpoint,returns,outcome,result,cost,latency
carol/papers,references,returned,5 documents,-,339 ms
erin/trials,references,returned,3 documents,$0.05,476 ms
frank/registry,references,rejectedrate limited,"Policy 'rate_limit' blocked request: {'limit': '2/m', 'remaining': 0} · resets in 60s",-,568 ms
dave/notes,references,returned,4 documents,$0.08,510 ms
olga/trials-assistant,both,returned,"2 documents · summary: Across my curated corpus, grade 3+ neutropenia is …",-,479 ms
source,policy,status,amount,wallet / rail,transaction
erin/trials,stripe_per_request,charged,$0.05,w-erin-stripe-usd,ledger-260113
dave/notes,xendit_per_document,charged,$0.08,w-dave-xendit-usd,ledger-126040


She reads what Erin's registry returned, and what it cost.

In [9]:
results["erin/trials"]

policy_type,status,amount,transaction / reason_code,
stripe_per_request,charged,$0.05,ledger-260113,


## Only recent work

Documents carry whatever metadata their Space attached. She keeps the ones published since 2024. This happens on her
side, over what came back, so it narrows the `limit` she already asked for; if she filtered hard she would raise it.
(The same verb on a plan, before executing, can send the filter to the Space instead: see *Advanced use cases*.)

In [10]:
recent = results.filter(published_gte="2024-01-01")
recent

endpoint,returns,outcome,result,cost,latency
carol/papers,references,returned,3 documents of 5 returned,-,339 ms
erin/trials,references,returned,2 documents of 3 returned,$0.05,476 ms
frank/registry,references,rejectedrate limited,"Policy 'rate_limit' blocked request: {'limit': '2/m', 'remaining': 0} · resets in 60s",-,568 ms
dave/notes,references,returned,3 documents of 4 returned,$0.08,510 ms
olga/trials-assistant,both,returned,"2 documents · summary: Across my curated corpus, grade 3+ neutropenia is …",-,479 ms
source,policy,status,amount,wallet / rail,transaction
erin/trials,stripe_per_request,charged,$0.05,w-erin-stripe-usd,ledger-260113
dave/notes,xendit_per_document,charged,$0.08,w-dave-xendit-usd,ledger-126040


## Talk it through

She picks a model to discuss the results with. A model is a Space endpoint too, so the chat opens with the same
pre-flight: price per message, her balance, a verdict. Ivan's model is metered and she has no credits there yet.
The card also says how much **context** every message will carry, because she pays for those tokens each turn.

In [11]:
chat = recent.chat("ivan/gpt-mini")
await chat.preflight()

Chat(models=['ivan/gpt-mini'], via=direct, turns=0)

Before spending anything she looks at exactly what the model will see. No turns yet, so only the passages: in citation
order, with Olga's own answer labelled as hers, so the model can cite a passage or another endpoint's conclusion.

In [12]:
chat.context

Context(0 earlier turns ~0 tokens, 10 references, 1 summaries ~249 tokens)

If she sends anyway, nothing goes to Ivan's Space. The reply is the same card, with the same action.

In [13]:
await chat.send("Summarise the serious adverse events across these sources.")

wallet,ivan · stripe USD w-ivan-stripe-usd
your balance,$0.00 · needed $0.02
bundles,starter: $5.00 pro: $20.00


In [14]:
invoice = await chat.top_up(bundle="starter")
world.pay(invoice)                                        # mock: paid at the link
await chat.send("Summarise the serious adverse events across these sources.")

[1],carol/papers
[2],carol/papers
[4],erin/trials


Citations point back to the context above. Now she follows up. The question only makes sense together with the
first turn, so the earlier turns ride along with every message; the reply says which turn this is and what the
conversation has cost so far.

In [15]:
await chat.send("Were any of them fatal?")

[3],carol/papers
[4],erin/trials


For the next question she only wants the registry and Olga's view, and fewer passages. `chat.use(view)` swaps the
passages; the conversation keeps its turns. `chat.context` shows both halves of what the next message will carry:
the earlier turns, then the passages in citation order, each with its token count.

In [16]:
chat.use(recent.only("erin/trials", "olga/trials-assistant").top(3))
chat.context

Context(2 earlier turns ~75 tokens, 3 references, 1 summaries ~116 tokens)

In [17]:
await chat.send("And in the registry data alone, what stood out?")

[1],erin/trials
[2],erin/trials


The model condenses its own previous answer: history, not context, is doing the work here.

In [18]:
await chat.send("Put that in two bullets for a slide.")

[1],erin/trials
[2],erin/trials


## A second opinion

Ivan is metered and she has paid for every message. Bob's model is free. A chat is a room: she adds Bob, who is
briefed with the questions so far and Ivan's answers, and from here every question goes to both. The card grows a
tab per model, each with its own pre-flight and transcript; the context line and the spend stay shared.

In [19]:
await chat.add("bob/llama-3")
chat

Chat(models=['ivan/gpt-mini', 'bob/llama-3'], via=direct, turns=5)

In [20]:
replies = await chat.send("Do you agree with those two bullets, and what would you add?")
replies

[4],olga/trials-assistant
[4],olga/trials-assistant


One reply per model, as tabs; `replies["bob/llama-3"]` picks one out. Bob is free and good enough, so she lets Ivan
go. His tab stays readable, he is just not asked again.

In [21]:
replies["bob/llama-3"].text

'Yes, both points hold against the passages in context; the registry and the pooled analysis agree on the SAE range and the dose-limiting toxicities. I would add that hepatic events stayed under 2% [4].'

In [22]:
await chat.remove("ivan/gpt-mini")

Chat(models=['bob/llama-3'], via=direct, turns=6)

## More evidence, mid-conversation

The consortium corpus was not in her search. A search is the same wherever she is: same pre-flight, same cost card.
She adds its results to what the chat already carries and checks the context before asking.

In [23]:
more = await hub.search("infection-related serious adverse events", sources=["heidi/shared-corpus"]).execute()
more

endpoint,returns,outcome,result,cost,latency
heidi/shared-corpus,references,returned,2 documents,$0.01,578 ms
source,policy,status,amount,wallet / rail,transaction
heidi/shared-corpus,cluster_per_request,charged,$0.01,w-consortium-cluster-usd,ledger-441499


In [24]:
chat.use(chat.view + more)
chat.context

Context(5 earlier turns ~235 tokens, 5 references, 1 summaries ~153 tokens)

In [25]:
await chat.send("Does the consortium data change the picture?")

[4],heidi/shared-corpus
[5],heidi/shared-corpus


The whole room, with each model's pre-flight at the top of its tab. Every message was one request per model asked;
the passages and the growing history are what she pays tokens for. `chat.reset()` would drop the earlier turns
while keeping the passages and the pre-flights.

In [26]:
chat

Chat(models=['bob/llama-3'], via=direct, turns=7)

## The other way round

Sometimes the question changes faster than the sources. `hub.chat(model, sources=...)` searches on every turn and then
asks; the same filters shape what each turn's results become before the model sees them. Free model, free sources, so
no pre-flight interrupts here.

In [27]:
live = hub.chat("bob/llama-3", sources=["carol/papers", "erin/trials"]).filter(published_gte="2024-01-01").top(4)
await live.send("How many deaths were reported, and were they related to treatment?")

[3],erin/trials


In [28]:
live.context      # what that turn carried; live.last.results is the search behind it

Context(1 earlier turns ~44 tokens, 4 references, 0 summaries ~92 tokens)

## Where this leaves us

- Browse, narrow, compose, execute, filter, chat: each step shows its cost before it spends anything.
- One search works over any endpoint. A data source returns references, a model a summary, some return both; the row
  says which, and the chat uses whatever came back.
- Payment is checked twice. The client predicts from Hub metadata and wallet balances and holds what it knows will fail;
  the Space's own 402 or 403 is the authority and lands in the same card. Only the second one costs a round trip.
- Search and chat behave the same way because a model is just another endpoint.
- What a chat sends is a *view of results* you can see (`chat.context`) and reshape with the same verbs you used on
  results (`filter`, `only`, `drop`, `top`), before the first paid message or between turns.
- A chat is a room. Questions are shared, each model keeps its own transcript, and the card is one tab per model.
  `chat.add` brings a model in from the next turn (briefed, by default), `chat.remove` stops asking one, `send`
  returns one reply per model, indexable by path, and `chat.spent` is the room's total.
- Search and chat compose both ways. `results.chat(...)` starts a chat from a search; `chat.use(chat.view + more)`
  folds a later search into a running chat. There is one way to search, and it always shows its cost first.
- Every message carries the earlier turns; the card shows how big that is, and `chat.reset()` starts over without
  losing the view.

*User flows* covers the rest: headless use, failures and retries, budgets, the Aggregator, errors and the sync twin.
*Advanced use cases* has the parts that go beyond today's Space API.